# Digit addition and its carry circuit

Train a small transformer on 2-digit decimal addition, then work out what algorithm it actually
learned.

Nanda et al. did this for modular arithmetic in *Progress measures for grokking via mechanistic
interpretability*, and found the model had built something out of discrete Fourier transforms and
trigonometric identities. Plain decimal addition is a different shape of problem. It's positional
and it needs carrying, so I'd expect a different circuit. Working out how different is the point.

The model is a TransformerLens `HookedTransformer` rather than my own implementation. That costs
nothing here and gives me hook points for the ablations later.

One thing makes this tractable: the vocabulary is 12 symbols, the digits plus `+` and `=`. In a
language model most of the parameters sit in the embedding table and never compute anything. Here
almost every parameter is doing work, so there should be a real algorithm to find rather than a
lookup table.

## The floor

Identify which attention heads implement the carry, using ablation and attention patterns.

Everything past that is bonus: the full mechanism, whether the circuit differs across digit
positions, the grokking dynamics. Interpretability has no natural stopping point, so it's worth
writing down in advance what counts as done.

In [ ]:
from dataclasses import dataclass

import einops
import torch as t
from jaxtyping import Int
from torch import Tensor
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer, HookedTransformerConfig

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
print("device:", device)

---

## 1. Task encoding

Sequences look like `49+97=641`, fixed length, so every example has the same shape. Twelve tokens
in the vocabulary: ten digits, `+`, and `=`. A dict covers it. There's no reason to train a
tokenizer for twelve symbols.

Two of the encoding choices matter more than they look.

The answer is written backwards, least significant digit first. Carries then propagate in the same
direction the model generates, so each answer digit depends on things it has already emitted.
Writing `146` the normal way round would ask it to predict the leading digit before it has worked
out the carries that determine it.

Numbers are zero-padded, `05` rather than `5`, so a given position always means the same thing.

Two digits is the smallest size that still needs carrying, and 100 x 100 = 10,000 pairs is small
enough to enumerate the whole problem.

In [ ]:
VOCABULARY = {f"{i:d}": i for i in range(10)}
VOCABULARY["+"] = 10
VOCABULARY["="] = 11

# convenience aliases
TOKEN_TO_ID = VOCABULARY
ID_TO_TOKEN = {idx: token for token, idx in TOKEN_TO_ID.items()}
D_VOCAB = len(VOCABULARY)

# 1D (seq,) or 2D (batch, seq) ids, as a tensor or plain lists
TokenIds = Tensor | list[int] | list[list[int]]


def to_tokens(strings: str | list[str]) -> Int[Tensor, "batch seq"]:
    """'49+97=641' -> tensor([[4, 9, 10, 9, 7, 11, 6, 4, 1]]). Always returns (batch, seq)."""
    if isinstance(strings, str):
        strings = [strings]
    return t.tensor([[TOKEN_TO_ID[c] for c in s] for s in strings], device=device)


def to_str_tokens(tokens: TokenIds) -> list[str] | list[list[str]]:
    """Token ids -> the characters they stand for, one per token. Nested if the input is 2D."""
    if isinstance(tokens, Tensor):
        tokens = tokens.tolist()
    if tokens and isinstance(tokens[0], list):
        return [[ID_TO_TOKEN[i] for i in row] for row in tokens]
    return [ID_TO_TOKEN[i] for i in tokens]


def to_string(tokens: TokenIds) -> str | list[str]:
    """Token ids -> '49+97=641'. A list of strings if the input is 2D."""
    strs = to_str_tokens(tokens)
    if strs and isinstance(strs[0], list):
        return ["".join(row) for row in strs]
    return "".join(strs)


# sequences will have the result of the computation reversed


## 2. Dataset

Enumerate every `(a, b)` pair, then split. 10,000 examples of 9 tokens is one tensor that fits in
memory, so there's no need for `datasets`, Arrow, or DataLoader workers.

The split fraction is the knob that matters most. Holding out a large fraction is what makes
grokking possible: the model memorises the training pairs, sits near chance on everything held
out, then generalises abruptly thousands of steps later. Reproducing that here would be a result
on its own, so the fraction is worth sweeping once a basic run works.

Loss is computed only on the answer positions. The question digits are given to the model, so
predicting them is free and would dilute the number.

In [ ]:
def sum_dataset(num_digits: int = 2, seed: int = 42, frac_train: float = 0.3) -> tuple[Tensor, Tensor]:
    
    pairs = []

    for i in range(10**num_digits):
        for j in range(10**num_digits):
            pairs.append(f"{i:0{num_digits}d}+{j:0{num_digits}d}={f'{(i+j):0{num_digits+1}d}'[::-1]}")

    data = t.tensor([[TOKEN_TO_ID[token] for token in example] for example in pairs], device=device)
    g = t.Generator().manual_seed(seed)

    _n = data.shape[0]
    split = t.randperm(_n, generator=g)
    train_idx = split[:int(frac_train * _n)].to(device=device)
    test_idx = split[int(frac_train * _n):].to(device=device)


    assert t.cat([train_idx, test_idx], dim=0).sort().values.equal(t.arange(len(data), device=data.device))

    train = data[train_idx]
    test = data[test_idx]
    
    return train, test
train, test = sum_dataset()
print(f"train: {train.shape} \ntest: {test.shape}")

## 3. Train

Sizing follows Nanda et al. wherever the task allows: `d_model=128`, 4 heads, `d_mlp=512`, no layer
norm. Where I deviate it's deliberate. Two layers rather than one, a 12-symbol vocabulary, and a
9-token context, all because multi-digit addition is longer and more compositional than a single
modular sum.

Training is full batch, AdamW, constant `lr=1e-3`, `weight_decay=1.0`. That last one isn't
incidental. Grokking depends on the decoupled weight decay specifically, and full batch keeps the
loss curves clean enough to see the transition when it happens.

### Training

I started from ARENA's `TrainArgs` and cut it down. Anything duplicating `cfg` went, since two
places holding the same number will eventually disagree, and so did everything that only makes
sense when batching through a DataLoader. Full batch means one epoch is one optimizer step, so a
single counter covers it.

In [ ]:
@dataclass
class TrainArgs:
    num_digits: int
    # trainset_size: int   -- derivable from train.shape[0]
    # valset_size: int     -- derivable from test.shape[0]
    frac_train: float
    epochs: int
    # batch_size: int      -- full batch, so there is nothing to vary
    lr: float
    # lr_start: float
    # lr_end: float
    weight_decay: float
    seed: int
    eval_every: int
    # d_model: int
    # d_head: int
    # n_layers: int
    # n_heads: int
    # d_mlp: int
    # normalization_type: str | None
    use_wandb: bool
    device: str

    def __post_init__(self):
        self.seq_len = self.num_digits * 3 + 2 + 1  # We have [{a}, +, {b}, =, {a+b}]

In [ ]:
args = TrainArgs(
    num_digits=2,
    frac_train=0.3,
    epochs=40000,
    lr=1e-3,
    weight_decay=1.0,
    seed=42,
    eval_every=10,
    use_wandb=False,
    device=device,
)

### Defining the model

In [ ]:
def create_model(args: TrainArgs) -> HookedTransformer:
    """A fresh model from init, sized from `args`.

    Re-seeded on every call, so a sweep over `frac_train` varies only the thing
    being swept -- every run starts from identical weights.
    """
    t.manual_seed(args.seed)

    d_model = 128
    n_heads = 4

    cfg = HookedTransformerConfig(
        n_layers=2,
        n_ctx=args.seq_len,  # We have [{a}, +, {b}, =, {a+b}]
        d_model=d_model,
        d_head=d_model // n_heads,
        n_heads=n_heads,
        d_mlp=4 * d_model,
        attn_only=False,
        act_fn="relu",
        # We have all digits, and "+="
        d_vocab=len(VOCABULARY),
        # it's a small transformer so may as well use these hooks
        use_attn_result=False,
        use_split_qkv_input=False,
        use_hook_tokens=False,
        normalization_type=None,
        device=args.device,
    )

    return HookedTransformer(cfg)

model = create_model(args)
cfg = model.cfg

In [ ]:
def param_breakdown(model: HookedTransformer, n_train: int | None = None) -> None:
    """Parameter counts grouped by component, with the attention:MLP ratio.

    The ratio is worth watching more than the total -- attention routes information
    between positions, MLPs do the arithmetic, and a config that starves the MLPs
    will struggle on this task. The standard d_mlp = 4 * d_model gives ~0.50x.
    """
    groups: dict[str, int] = {}
    for name, param in model.named_parameters():
        key = "attn" if ".attn." in name else "mlp" if ".mlp." in name else "embed/unembed"
        groups[key] = groups.get(key, 0) + param.numel()

    total = sum(groups.values())
    for key, count in sorted(groups.items(), key=lambda kv: -kv[1]):
        print(f"  {key:16s} {count:>10,}  ({count / total * 100:5.1f}%)")
    print(f"  {'total':16s} {total:>10,}")

    if groups.get("mlp"):
        print(f"\n  attn:mlp ratio   {groups['attn'] / groups['mlp']:.2f}x")
    if n_train:
        print(f"  params per training example   {total / n_train:.0f}")


param_breakdown(model)


In [ ]:
for name, param in model.named_parameters():
    print(f"{name:<20} {str(param.shape):<20} {param.numel()}")
sum(p.numel() for p in model.parameters())      # 584,556

In [ ]:
print(
    train.shape,
    to_string(train[t.randint(low=0, high=len(train), size=(1,))]),
    sep="\n",
)

In [ ]:
class Trainer:
    def __init__(self, model: HookedTransformer, args: TrainArgs):
        self.args = args
        self.model = model
        
    def training_step(self, toks: Tensor) -> Tensor:
        logits, target = self._shared_train_validation_step(toks)
        return self._loss(logits, target)
    
    def validation_step(self, toks: Tensor) -> tuple[float, float]:
        logits, target = self._shared_train_validation_step(toks)
        loss = self._loss(logits, target).item()
        accuracy = (logits.argmax(dim=-1) == target).float().mean().item()
        return loss, accuracy
    
    def _loss(self, logits: Tensor, target: Tensor) -> Tensor:
        return t.nn.functional.cross_entropy(
            einops.rearrange(logits, "batch seq vocab_out -> (batch seq) vocab_out"),
            einops.rearrange(target, "batch seq -> (batch seq)")
        )
    
    def _shared_train_validation_step(self, toks: Tensor) -> tuple[Tensor, Tensor]:
        toks = toks.to(self.args.device)
        logits = self.model(toks)[:, -(self.args.num_digits + 2): -1]
        target = toks[:, -(self.args.num_digits + 1):]
        return logits, target
    
    def configure_optimizers(self):
        optimizer = t.optim.AdamW(
            self.model.parameters(), lr=self.args.lr, weight_decay=self.args.weight_decay
        )
        return optimizer

def train_model(model: HookedTransformer, args: TrainArgs):
    trainer = Trainer(model=model, args=args)
    optimizer = trainer.configure_optimizers()
    
    train, test = sum_dataset(
        num_digits=args.num_digits, seed=args.seed, frac_train=args.frac_train
    )

    # the two curves that make the grokking plot, plus test accuracy
    history = {"step": [], "train_loss": [], "test_loss": [], "test_accuracy": []}

    # training
    optimizer.zero_grad()
    progress_bar = tqdm(range(args.epochs))
    for epoch in progress_bar:
        loss = trainer.training_step(train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
        # validation
        if epoch % args.eval_every == 0 or epoch == args.epochs - 1:
            with t.inference_mode():
                test_loss, test_accuracy = trainer.validation_step(test)
            history["step"].append(epoch)
            history["train_loss"].append(loss.item())
            history["test_loss"].append(test_loss)
            history["test_accuracy"].append(test_accuracy)
            progress_bar.set_description(
                f"train {loss.item():.4f} | test {test_loss:.4f} | acc {test_accuracy:.3f}"
            )

    return trainer.model, history


## 4. Interpretability

Roughly in order of cost:

- Attention patterns on carry versus non-carry examples, via `circuitsvis`. Do particular heads
  look at the digit positions that generate a carry?
- Per-head ablation. Zero `hook_z` for one head at a time and measure the accuracy drop. That
  tells me which heads matter, not how they work.
- Direct logit attribution, to find which component actually writes the answer digit.
- Then the mechanism: how carry information moves between digit positions.

Method reference is ARENA's exercise on the Nanda result, *1.5.2 Grokking & Modular Arithmetic*.

The obvious trap is attention sinks. These sequences have no BOS token, so a head with nothing
useful to attend to will dump its attention on whatever sits at position 0, or on the `=`. If a
head looks like it attends to the equals sign for no reason I can explain, it's probably a sink
and not part of any circuit.

In [ ]:
model = create_model(args)
trainer = Trainer(model, args)

train_model(model, args)

In [ ]:
import sys, json

tb = getattr(sys, "last_traceback", None)
if tb is None:
    raise RuntimeError("no traceback - has the run actually been interrupted?")

history = None
while tb:
    if tb.tb_frame.f_code.co_name == "train_model":
        history = tb.tb_frame.f_locals["history"]
        break
    tb = tb.tb_next

if history is None:
    raise RuntimeError("train_model not in the traceback - wrong exception, or the cell was re-run")

json.dump(history, open("../results/history_wd1.0.json", "w"))
print(f"{len(history['step'])} eval points, last step {history['step'][-1]}, "
      f"final acc {history['test_accuracy'][-1]:.3f}")


## 5. Write-up

Goes in `README.md` and `results/`: motivation, method, findings, plots. The comparison that
matters is the circuit here against Nanda's modular-arithmetic one.